# Scene-Level Context Notebook

Build scene-level MCQ + FRQ examples from nuScenes `scene.description` using weak rule-based labels.

Included tasks:
- `SP-C-1`: weather / environmental condition
- `SP-C-2`: time of day
- `SP-C-3`: road type
- `SP-C-4`: traffic density / crowdedness
- `SP-C-5`: baseline scene risk level
- `SP-C-6`: overall scene caption (FRQ)

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '6, 7'
DATAROOT = Path('/home/rgao727/Autonomous Driving/NuScenes-QA/data/nuscenes-v1.0-mini/v1.0-mini')
SCENE_JSON = DATAROOT / 'scene.json'
LOG_JSON = DATAROOT / 'log.json'

with SCENE_JSON.open('r', encoding='utf-8') as f:
    scenes = json.load(f)

with LOG_JSON.open('r', encoding='utf-8') as f:
    logs = json.load(f)

print(f'Loaded {len(scenes)} scenes and {len(logs)} logs.')

In [ ]:
def desc(text: str) -> str:
    return str(text or '').strip().lower()


def infer_weather_choice(description: str):
    d = desc(description)
    if any(k in d for k in ['fog', 'haze', 'mist']):
        return 'E', 'Fog / Heavy haze'
    if any(k in d for k in ['snow', 'ice']):
        return 'D', 'Snow / Ice'
    if any(k in d for k in ['cloudy', 'overcast']):
        return 'C', 'Overcast / Cloudy'
    if any(k in d for k in ['rain', 'after rain', 'wet']):
        return 'B', 'Rain / Wet road surface'
    return 'A', 'Clear / Sunny'


def infer_time_of_day_choice(description: str):
    d = desc(description)
    if 'night' in d:
        return 'B', 'Nighttime (relying on streetlights/headlights)'
    if any(k in d for k in ['dawn', 'dusk', 'sunrise', 'sunset', 'glare']):
        return 'C', 'Dawn / Dusk (transition lighting, potential glare)'
    return 'A', 'Daytime (adequate natural lighting)'


def infer_road_type_choice(description: str):
    d = desc(description)
    if any(k in d for k in ['parking lot', 'off-road', 'off road']):
        return 'E', 'Parking lot / Off-road area'
    if 'construction' in d:
        return 'D', 'Construction zone'
    if any(k in d for k in ['high speed', 'highway', 'expressway']):
        return 'C', 'Highway / Expressway'
    if any(k in d for k in ['intersection', 'crossroad', 'cross intersection', 'crosswalk']):
        return 'A', 'Urban intersection / Crossroad'
    return 'B', 'Straight urban/suburban road'


def infer_density_choice(description: str):
    d = desc(description)
    score = 0
    score += 3 if 'many peds' in d else 0
    score += 2 if 'peds' in d else 0
    score += 2 if 'jaywalker' in d else 0
    score += 1 if 'cyclist' in d or 'bicycle' in d else 0
    score += 1 if 'scooter' in d or 'pmd' in d else 0
    score += 1 if 'bus' in d else 0
    score += 1 if 'truck' in d else 0
    score += 1 if 'construction vehicle' in d else 0
    score += 1 if 'cars' in d else 0
    score += 1 if 'busy' in d else 0
    if score >= 6:
        return 'C', 'Dense'
    if score >= 3:
        return 'B', 'Moderate'
    return 'A', 'Sparse'


def infer_risk_choice(description: str):
    d = desc(description)
    score = 0
    score += 2 if 'night' in d else 0
    score += 1 if 'after rain' in d or 'wet' in d else 0
    score += 2 if 'jaywalker' in d else 0
    score += 1 if 'many peds' in d else 0
    score += 1 if 'peds crossing crosswalk' in d or 'crossing crosswalk' in d else 0
    score += 1 if 'difficult lighting' in d else 0
    score += 1 if 'construction' in d else 0
    score += 1 if 'high speed' in d else 0
    score += 1 if 'truck' in d or 'bus' in d else 0
    if score >= 6:
        return 'C', 'High risk'
    if score >= 3:
        return 'B', 'Moderate risk'
    return 'A', 'Low risk'


def build_scene_caption(description: str) -> str:
    d = desc(description)
    weather_key, weather_text = infer_weather_choice(description)
    tod_key, tod_text = infer_time_of_day_choice(description)
    road_key, road_text = infer_road_type_choice(description)
    density_key, density_text = infer_density_choice(description)

    first = []
    if tod_key == 'B':
        first.append('at night')
    elif tod_key == 'C':
        first.append('during a dawn or dusk transition')
    else:
        first.append('in daytime')

    if weather_key == 'B':
        first.append('with wet or post-rain conditions')
    elif weather_key == 'C':
        first.append('under overcast or cloudy conditions')
    elif weather_key == 'D':
        first.append('with snow or icy conditions')
    elif weather_key == 'E':
        first.append('under foggy or hazy visibility')

    sentence1 = 'This scene takes place ' + ' '.join(first) + '.'
    sentence2 = f'The driving environment is best described as {road_text.lower()} with {density_text.lower()} surrounding traffic activity.'

    notable = []
    for key, phrase in [
        ('jaywalker', 'a jaywalker'),
        ('many peds', 'many pedestrians'),
        ('peds', 'pedestrians'),
        ('truck', 'a truck'),
        ('bus', 'a bus'),
        ('scooter', 'a scooter'),
        ('pmd', 'a PMD rider'),
        ('bicycle', 'a bicycle'),
        ('cyclist', 'a cyclist'),
        ('construction', 'construction activity'),
    ]:
        if key in d and phrase not in notable:
            notable.append(phrase)
    if notable:
        sentence3 = 'Notable scene elements include ' + ', '.join(notable[:-1]) + (f', and {notable[-1]}.' if len(notable) > 1 else f'{notable[0]}.')
        return ' '.join([sentence1, sentence2, sentence3])
    return ' '.join([sentence1, sentence2])


In [ ]:
MCQ_DEFS = [
    {
        'id': 'SP-C-1',
        'task': 'scene-context',
        'question_format': 'MCQ',
        'question': 'What is the primary weather and environmental condition in the current scene?',
        'choices': {
            'A': 'Clear / Sunny',
            'B': 'Rain / Wet road surface',
            'C': 'Overcast / Cloudy',
            'D': 'Snow / Ice',
            'E': 'Fog / Heavy haze'
        },
        'infer': infer_weather_choice,
    },
    {
        'id': 'SP-C-2',
        'task': 'scene-context',
        'question_format': 'MCQ',
        'question': 'Based on the lighting conditions and sky visibility, what is the estimated time of day?',
        'choices': {
            'A': 'Daytime (adequate natural lighting)',
            'B': 'Nighttime (relying on streetlights/headlights)',
            'C': 'Dawn / Dusk (transition lighting, potential glare)'
        },
        'infer': infer_time_of_day_choice,
    },
    {
        'id': 'SP-C-3',
        'task': 'scene-context',
        'question_format': 'MCQ',
        'question': 'Which of the following best describes the overall road typology and driving environment?',
        'choices': {
            'A': 'Urban intersection / Crossroad',
            'B': 'Straight urban/suburban road',
            'C': 'Highway / Expressway',
            'D': 'Construction zone',
            'E': 'Parking lot / Off-road area'
        },
        'infer': infer_road_type_choice,
    },
    {
        'id': 'SP-C-4',
        'task': 'scene-context',
        'question_format': 'MCQ',
        'question': 'How spatially dense or crowded is the current traffic environment considering all visible road users?',
        'choices': {
            'A': 'Sparse',
            'B': 'Moderate',
            'C': 'Dense'
        },
        'infer': infer_density_choice,
    },
    {
        'id': 'SP-C-5',
        'task': 'scene-context',
        'question_format': 'MCQ',
        'question': 'What is the baseline safety risk level of the overall scene, independent of any single specific object?',
        'choices': {
            'A': 'Low risk',
            'B': 'Moderate risk',
            'C': 'High risk'
        },
        'infer': infer_risk_choice,
    },
]

FRQ_DEF = {
    'id': 'SP-C-6',
    'task': 'scene-context',
    'question_format': 'FRQ',
    'question': 'Describe the overall environment of the scene, including weather, lighting conditions, road type, and notable surrounding activity.',
}


def build_scene_tasks(scene_row: dict):
    description = str(scene_row.get('description', '')).strip()
    scene_name = str(scene_row.get('name', ''))
    out = []
    for mcq in MCQ_DEFS:
        key, text = mcq['infer'](description)
        out.append({
            'id': mcq['id'],
            'task': mcq['task'],
            'question_format': mcq['question_format'],
            'scene_id': scene_name,
            'scene_description': description,
            'question': mcq['question'],
            'choices': mcq['choices'],
            'ground_truth': key,
            'ground_truth_text': text,
            'model_response': '',
        })
    out.append({
        'id': FRQ_DEF['id'],
        'task': FRQ_DEF['task'],
        'question_format': FRQ_DEF['question_format'],
        'scene_id': scene_name,
        'scene_description': description,
        'question': FRQ_DEF['question'],
        'ground_truth': build_scene_caption(description),
        'model_response': '',
    })
    return out

In [ ]:
scene_task_groups = []
all_tasks = []
for scene_row in scenes:
    tasks = build_scene_tasks(scene_row)
    scene_task_groups.append({'scene_id': scene_row['name'], 'tasks': tasks})
    all_tasks.extend(tasks)

print(f'Built {len(all_tasks)} tasks across {len(scene_task_groups)} scenes.')
all_tasks[:6]

In [ ]:
rows = []
for task in all_tasks:
    row = {
        'scene_id': task['scene_id'],
        'id': task['id'],
        'question_format': task['question_format'],
        'question': task['question'],
        'ground_truth': task['ground_truth'],
    }
    if 'ground_truth_text' in task:
        row['ground_truth_text'] = task['ground_truth_text']
    rows.append(row)

if pd is not None:
    df = pd.DataFrame(rows)
    df
else:
    rows[:10]

In [ ]:
# Inspect a few full scene-level task bundles.
for bundle in scene_task_groups[:3]:
    print('=' * 110)
    print('scene_id:', bundle['scene_id'])
    print('scene_description:', bundle['tasks'][0]['scene_description'])
    for task in bundle['tasks']:
        print('-', task['id'], '|', task['question'])
        print('  GT:', task['ground_truth'])
        if 'ground_truth_text' in task:
            print('  GT text:', task['ground_truth_text'])

In [ ]:
# Distribution check for the weak labels.
for task_id in ['SP-C-1', 'SP-C-2', 'SP-C-3', 'SP-C-4', 'SP-C-5']:
    dist = Counter([t['ground_truth'] for t in all_tasks if t['id'] == task_id])
    print(task_id, dict(dist))

In [ ]:
OUT_JSON = Path('/home/rgao727/Autonomous Driving/scene-level-context-tasks.json')
with OUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(all_tasks, f, ensure_ascii=False, indent=2)
print(f'Saved {len(all_tasks)} tasks to {OUT_JSON}')

In [ ]:
# ===== Qwen Model Setup =====
# Same style as spatial_understanding.ipynb
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

my_cache_dir = '/data2/rgao727/hf_cache_store'

try:
    os.makedirs(my_cache_dir, exist_ok=True)
    print(f'Successfully created: {my_cache_dir}')
except PermissionError:
    print('Permission denied for /data2, falling back to /home cache directory.')
    my_cache_dir = os.path.expanduser('~/hf_cache_store_backup')
    os.makedirs(my_cache_dir, exist_ok=True)

hf_token = '#TODO'
min_pixels = 256 * 28 * 28
max_pixels = 768 * 28 * 28
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-30B-A3B-Instruct',
    trust_remote_code=True,
    token=hf_token,
    cache_dir=my_cache_dir,
    min_pixels=min_pixels,
    max_pixels=max_pixels,
)
model = AutoModelForImageTextToText.from_pretrained(
    'Qwen/Qwen3-VL-30B-A3B-Instruct',
    device_map='auto',
    torch_dtype=torch.bfloat16,
    token=hf_token,
    attn_implementation='flash_attention_2',
    cache_dir=my_cache_dir,
)

In [ ]:
# ===== Lightweight text-only evaluation using the same model =====
# Here we test metadata-grounded answering first. MCQ accuracy is exact-match on option key.
EVAL_LIMIT = 12
MAX_NEW_TOKENS = 180
eval_tasks = all_tasks[:EVAL_LIMIT]
print(f'Evaluating {len(eval_tasks)} tasks')

def _apply_chat(processor, messages):
    try:
        return processor.apply_chat_template(
            conversation=messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
        )

def _clean_text(s):
    s = str(s).replace('<|im_end|>', '').strip()
    return ' '.join(s.split())

def _extract_option(text):
    m = re.search(r'\b([A-E])\b', _clean_text(text).upper())
    return m.group(1) if m else None

def _keyword_set(text):
    words = re.findall(r'[a-z]+', str(text).lower())
    stop = {'the', 'a', 'an', 'of', 'and', 'in', 'at', 'with', 'this', 'scene', 'is', 'to', 'for'}
    return {w for w in words if w not in stop and len(w) > 2}

@torch.inference_mode()
def infer_answer(messages):
    inputs = _apply_chat(processor, messages)
    dev = model.get_input_embeddings().weight.device
    inputs = {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    pred = processor.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return _clean_text(pred)

rows = []
for i, task in enumerate(eval_tasks, 1):
    prompt = f"Scene description: {task['scene_description']}\n\nQuestion: {task['question']}\n"
    if task['question_format'] == 'MCQ':
        prompt += f"Choices: {json.dumps(task['choices'])}\nRespond with only the option key."
    else:
        prompt += 'Answer concisely in 2-3 sentences. Do not invent details beyond the description.'
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': prompt}]}]
    pred = infer_answer(messages)
    row = {
        'scene_id': task['scene_id'],
        'id': task['id'],
        'question_format': task['question_format'],
        'ground_truth': task['ground_truth'],
        'prediction': pred,
    }
    if task['question_format'] == 'MCQ':
        pred_key = _extract_option(pred)
        row['predicted_option'] = pred_key
        row['correct'] = pred_key == task['ground_truth']
    else:
        gt_keywords = _keyword_set(task['ground_truth'])
        pred_keywords = _keyword_set(pred)
        overlap = sorted(gt_keywords & pred_keywords)
        row['keyword_overlap'] = ', '.join(overlap)
        row['keyword_recall'] = len(overlap) / len(gt_keywords) if gt_keywords else None
    rows.append(row)
    print(f"[{i}/{len(eval_tasks)}] {task['id']} {task['scene_id']} done")

if pd is not None:
    eval_df = pd.DataFrame(rows)
    eval_df
else:
    rows